In [1]:
from google.colab import drive
drive.mount('/content/drive')
import os
os.chdir('/content/drive/MyDrive/Deep Learning Project/DLE-Flair-Segmentation')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [2]:
import torch
from torch.utils.data import Dataset, DataLoader
from pathlib import Path
import numpy as np
from PIL import Image
import csv

In [3]:
splits_dir = Path("data/processed/splits")

In [4]:
def load_split(csv_path):
    records = []
    base_dir = Path.cwd() # Get the current working directory (project root)
    with csv_path.open() as f:
        reader = csv.reader(f)
        next(reader, None) # Skip header
        for row in reader:
            if len(row) >= 2:
                image_rel, mask_rel = row[0], row[1]
                records.append({
                    "image_path": (base_dir / image_rel).resolve(),
                    "mask_path": (base_dir / mask_rel).resolve(),
                })
            else:
                print(f"Warning: Skipping malformed row in {csv_path} with content: {row}")
    return records

In [5]:
train_records = load_split(splits_dir / "grouped_5class_train.csv")
val_records   = load_split(splits_dir / "grouped_5class_val.csv")
test_records  = load_split(splits_dir / "grouped_5class_test.csv")

print(f"Train: {len(train_records)}")
print(f"Val:   {len(val_records)}")
print(f"Test:  {len(test_records)}")

Train: 200
Val:   25
Test:  25


In [6]:
class FlairDataset(Dataset):
    def __init__(self, records, transform=None, remap_classes=False):
        self.records = records
        self.transform = transform
        #self.remap_classes = remap_classes

    def __len__(self):
        return len(self.records)

    def __getitem__(self, idx):
        record = self.records[idx]

        image = np.array(Image.open(record["image_path"]).convert("RGB"))
        mask = np.array(Image.open(record["mask_path"]))


        image = torch.tensor(image).permute(2, 0, 1).float() / 255.0
        mask = torch.tensor(mask).long()

        return image, mask

In [7]:
train_dataset = FlairDataset(train_records, remap_classes=True)
val_dataset   = FlairDataset(val_records, remap_classes=True)
test_dataset  = FlairDataset(test_records, remap_classes=True)

In [8]:
print(f"Train dataset size: {len(train_dataset)}")
print(f"Validation dataset size: {len(val_dataset)}")
print(f"Test dataset size: {len(test_dataset)}")

Train dataset size: 200
Validation dataset size: 25
Test dataset size: 25


In [9]:
train_loader = DataLoader(train_dataset, batch_size=8, shuffle=True)
val_loader   = DataLoader(val_dataset, batch_size=8, shuffle=False)
test_loader  = DataLoader(test_dataset, batch_size=8, shuffle=False)